In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import  BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [14]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.5 
)

In [15]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [16]:
def chat_node(state: ChatState):
    messages = state['messages']

    res = llm.invoke(messages)

    return {'messages': [res]}

In [20]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [21]:
initial_state = {
    "messages": [
        HumanMessage(content="What is the capital of pakistan")
    ]
}

final_state = chatbot.invoke(initial_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [18]:
final_state

{'messages': [HumanMessage(content='What is the capital of pakistan', additional_kwargs={}, response_metadata={}, id='fbdce8d5-0047-45bb-9993-39751a1ab606'),
  AIMessage(content='The capital of Pakistan is **Islamabad**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fd14f-9a89-77b3-a6a9-5d43b28b21d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 79, 'total_tokens': 86, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 70}})]}

In [23]:
thread_id = '1'

while True:
    user_input = input("Enter message")

    if user_input.strip().lower() in ['quit', 'bye', 'exit']:
        break
    current_state = {
     "messages": [
        HumanMessage(content=user_input)
     ]
    }
    
    config = {
        'configurable': {"thread_id": thread_id}
    }
    res = chatbot.invoke(current_state, config=config)
    print("You: ", user_input)
    print('AI: ', res['messages'][-1].content )



You:  Hi my name is luffy
AI:  Hi Luffy! It's great to meet you. How can I help you today?
You:  what is my name
AI:  Your name is **Luffy**!
